## Databricks Streaming Pipeline
Aiven Kafka -> PySpark Structured Streaming -> Neo4j AuraDB -> Snowflake

| Step | Description |
|------|-------------|
| 0 | Install dependencies |
| 1 | Load credentials and configuration |
| 2 | Verify ca.pem certificate |
| 3 | Neo4j connect and define helpers |
| 4 | Snowflake connect and verify schema |
| 5 | Kafka define streaming reader |
| 6 | Define JSON schema and parse stream |
| 7 | Define `process_batch` enrichment function |
| 8 | Start live streaming pipeline |
| 9 | Monitor and stop stream |

Run each step cell and its debug cell before moving to the next.

### Step 0 - Install Python Dependencies

In [0]:
# Step 0 - Install neo4j Python driver
%pip install neo4j

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


#### Debug 0 - Confirm neo4j installed

In [0]:
# Debug 0 - Confirm neo4j installed correctly
# Expected: version number printed, no ImportError
import neo4j
from neo4j import GraphDatabase

print(f'neo4j version: {neo4j.__version__}')
print('GraphDatabase import OK')

neo4j version: 6.2.0
GraphDatabase import OK


#### Step 0b - Install python-dotenv


In [0]:
# Step 0b - Install python-dotenv so we can load pharma_pipeline.env
%pip install python-dotenv --quiet


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### Step 1 - Credentials and Configuration

#### Step 1a - Load pharma_pipeline.env into environment variables
This file lives in the Workspace root, not in this Drafts notebook's folder, so it has to be loaded explicitly - Databricks does not auto-source `.env` files.


In [0]:
# Step 1a - Load pharma_pipeline.env before reading any secrets below.
# This must run BEFORE the credential-loading cell, or KAFKA_USERNAME / KAFKA_PASSWORD
# (and the other keys in the file) will still be missing from os.environ.
from dotenv import load_dotenv
import os

# Update this if pharma_pipeline.env lives somewhere else in your Workspace.
# Based on the ca.pem path used later in this notebook, it's likely in the same
# Workspace user folder:
ENV_FILE_PATH = "/Workspace/Users/ma30412192101612@depi.eui.edu.eg/DataDose/pharma_pipeline.env"

if os.path.exists(ENV_FILE_PATH):
    load_dotenv(ENV_FILE_PATH, override=True)
    print(f'Loaded environment variables from {ENV_FILE_PATH}')
else:
    print(f'WARNING: .env file not found at {ENV_FILE_PATH}')
    print('Update ENV_FILE_PATH above to the correct Workspace location of pharma_pipeline.env.')
    print("Falling back to the Databricks secret scope 'datadose' for any missing values.")


Loaded environment variables from /Workspace/Users/ma30412192101612@depi.eui.edu.eg/DataDose/pharma_pipeline.env


In [0]:
# Step 1 - Load credentials and configuration
import os

def get_env(name, default=None, required=False):
    value = os.getenv(name)
    if value not in (None, ''):
        return value
    if default is not None:
        return default
    if required:
        raise RuntimeError(f'Missing required environment variable: {name}')
    return ''

def get_secret_or_env(env_name, secret_scope, secret_key, default=None, required=False):
    value = os.getenv(env_name)
    if value not in (None, ''):
        return value
    dbutils_obj = globals().get('dbutils')
    if dbutils_obj is not None:
        try:
            return dbutils_obj.secrets.get(scope=secret_scope, key=secret_key)
        except Exception:
            pass
    if default is not None:
        return default
    if required:
        raise RuntimeError(
            f'Missing required secret. Set {env_name} or add {secret_key} to the Databricks secret scope.'
        )
    return ''

# Aiven Kafka
# Credentials pulled from Databricks secret scope 'datadose'. No hardcoded fallback -
# fails fast if the scope/keys aren't set up yet.
KAFKA_BOOTSTRAP = get_env('KAFKA_BOOTSTRAP_SERVERS', 'datadosekafka-901-datadosedepiproject001.l.aivencloud.com:15816')
KAFKA_TOPIC = get_env('KAFKA_TOPIC', 'DataDose.in')
KAFKA_TOPIC_OUT = get_env('KAFKA_TOPIC_OUT', 'DataDose.out')
KAFKA_GROUP_ID = get_env('KAFKA_GROUP_ID', 'PySparkDataBricks-group')
# NOTE: 'dbfs:/...' paths fail with DBFS_DISABLED on workspaces where the public DBFS
# root is turned off (common on UC-governed workspaces, serverless or classic). Unity
# Catalog Volumes are the supported replacement for both plain file writes AND
# Structured Streaming checkpointLocation - see the pharma_pipeline_data volume created
# in Step 1b.
DEAD_LETTER_PATH = get_env('DEAD_LETTER_PATH', '/Volumes/ali_vm/pharma/pharma_pipeline_data/dead_letter')
KAFKA_USERNAME = get_secret_or_env('KAFKA_USERNAME', 'datadose', 'kafka-username', required=True)
KAFKA_PASSWORD = get_secret_or_env('KAFKA_PASSWORD', 'datadose', 'kafka-password', required=True)
KAFKA_CA_PEM_PATH = get_env('KAFKA_CA_PEM_PATH', required=True)

# Neo4j AuraDB
NEO4J_URI = get_env('NEO4J_URI', required=True)
NEO4J_USER = get_secret_or_env('NEO4J_USER', 'datadose', 'neo4j-user', required=True)
NEO4J_PASSWORD = get_secret_or_env('NEO4J_PASSWORD', 'datadose', 'neo4j-password', required=True)

# Snowflake
# NOTE: 'sfURL' is on Databricks' serverless write-options BLOCKLIST
# ([DATA_SOURCE_OPTIONS_VALIDATION_FAILED.SERVERLESS_WRITE_OPTIONS_NOT_ALLOWED]).
# 'host' / 'port' / 'sfAccount' are supported instead and work on both serverless and
# classic compute, so we derive them from SNOWFLAKE_URL instead of passing sfURL directly.
def _parse_snowflake_url(url: str) -> dict:
    cleaned = url.strip().replace('https://', '').replace('http://', '').rstrip('/')
    if ':' in cleaned:
        host, port = cleaned.split(':', 1)
    else:
        host, port = cleaned, '443'
    account = host.split('.snowflakecomputing.com')[0]
    return {'host': host, 'port': port, 'sfAccount': account}

SNOWFLAKE_SOURCE = 'net.snowflake.spark.snowflake'
sf_options = {
    **_parse_snowflake_url(get_env('SNOWFLAKE_URL', required=True)),
    'sfUser'      : get_secret_or_env('SNOWFLAKE_USER', 'pharma-snowflake', 'snowflake-user', required=True),
    'sfPassword'  : get_secret_or_env('SNOWFLAKE_PASSWORD', 'pharma-snowflake', 'snowflake-password', required=True),
    'sfDatabase'  : get_env('SNOWFLAKE_DATABASE', 'PHARMA_ANALYTICS_DB'),
    'sfSchema'    : get_env('SNOWFLAKE_SCHEMA', 'STAGING'),
    'sfWarehouse' : get_env('SNOWFLAKE_WAREHOUSE', 'PHARMA_WH'),
    'sfRole'      : get_env('SNOWFLAKE_ROLE', 'PYSPARK_ROLE'),
}

# Kafka JAAS config
# The kafkashaded prefix is required on the Spark JVM classpath.
KAFKA_JAAS = (
    f'kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required '
    f'username=\"{KAFKA_USERNAME}\" password=\"{KAFKA_PASSWORD}\";'
)

print('All config variables set.')
print(f'  Input topic  : {KAFKA_TOPIC}')
print(f'  Output topic : {KAFKA_TOPIC_OUT}')
print(f'  Dead letter  : {DEAD_LETTER_PATH}')


All config variables set.
  Input topic  : DataDose.in
  Output topic : DataDose.out
  Dead letter  : /Volumes/ali_vm/pharma/pharma_pipeline_data/dead_letter


#### Debug 1 - Print config (passwords masked)

In [0]:
# Debug 1 - Print config with passwords masked
def mask(s: str) -> str:
    return s[:4] + '****' + s[-4:] if len(s) > 8 else '****'

print('Kafka')
print(f'  Bootstrap   : {KAFKA_BOOTSTRAP}')
print(f'  Topic       : {KAFKA_TOPIC}')
print(f'  Username    : {KAFKA_USERNAME}')
print(f'  Password    : {mask(KAFKA_PASSWORD)}')
print(f'  ca.pem path : {KAFKA_CA_PEM_PATH}')
print()
print('Neo4j')
print(f'  URI      : {NEO4J_URI}')
print(f'  User     : {NEO4J_USER}')
print(f'  Password : {mask(NEO4J_PASSWORD)}')
print()
print('Snowflake')
for k, v in sf_options.items():
    display_v = mask(str(v)) if 'Password' in k or 'password' in k else v
    print(f'  {k:<15}: {display_v}')
print()
print('JAAS (first 60 chars)')
print(f'  {KAFKA_JAAS[:60]}...')

Kafka
  Bootstrap   : datadosekafka-901-datadosedepiproject001.l.aivencloud.com:15816
  Topic       : DataDose.in
  Username    : avnadmin
  Password    : AVNS****zFPu
  ca.pem path : /Workspace/Users/aa30503122102331@depi.eui.edu.eg/ca.pem

Neo4j
  URI      : neo4j+s://95a45d6f.databases.neo4j.io
  User     : neo4j
  Password : T5GW****EMt0

Snowflake
  host           : TLFCMYB-UJ75221.snowflakecomputing.com
  port           : 443
  sfAccount      : TLFCMYB-UJ75221
  sfUser         : PYSPARK_SVC
  sfPassword     : Data****025!
  sfDatabase     : PHARMA_ANALYTICS_DB
  sfSchema       : STAGING
  sfWarehouse    : PHARMA_WH
  sfRole         : PYSPARK_ROLE

JAAS (first 60 chars)
  kafkashaded.org.apache.kafka.common.security.scram.ScramLogi...


#### Step 1b - Create UC Volume for certificates (one-time setup)
Shared clusters require SSL truststore/keystore files to live in a Unity Catalog Volume or external location - a plain `/tmp` path is rejected. This only needs to run once; `CREATE ... IF NOT EXISTS` makes it safe to re-run.


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS datadose_workspace.pharma;

CREATE VOLUME IF NOT EXISTS datadose_workspace.pharma.pharma_certs;

CREATE VOLUME IF NOT EXISTS datadose_workspace.pharma.pharma_pipeline_data;

### Step 2 - Verify ca.pem Certificate

In [0]:
import os

KAFKA_CA_PEM_PATH = "/Volumes/datadose_workspace/pharma/pharma_certs/ca.pem"

if os.path.exists(KAFKA_CA_PEM_PATH):
    print(f"✓ ca.pem found: {KAFKA_CA_PEM_PATH}")
else:
    raise FileNotFoundError(
        f"ca.pem not found at {KAFKA_CA_PEM_PATH}"
    )

# Spark will use this path directly
KAFKA_CA_PEM_SPARK = KAFKA_CA_PEM_PATH

✓ ca.pem found: /Volumes/datadose_workspace/pharma/pharma_certs/ca.pem


#### Debug 2 - Inspect ca.pem content

In [0]:
# Debug 1 - Print config with passwords masked
def mask(s: str) -> str:
    if not s:
        return '****'
    return s[:4] + '****' + s[-4:] if len(s) > 8 else '****'

print('Kafka')
print(f'  Bootstrap   : {KAFKA_BOOTSTRAP}')
print(f'  Topic       : {KAFKA_TOPIC}')
print(f'  Username    : {mask(KAFKA_USERNAME)}')
print(f'  Password    : {mask(KAFKA_PASSWORD)}')
print(f'  ca.pem path : {KAFKA_CA_PEM_PATH}')
print()
print('Neo4j')
print(f'  URI      : {NEO4J_URI}')
print(f'  User     : {mask(NEO4J_USER)}')
print(f'  Password : {mask(NEO4J_PASSWORD)}')
print()
print('Snowflake')
for k, v in sf_options.items():
    display_v = mask(str(v)) if k in {'sfUser', 'sfPassword'} else v
    print(f'  {k:<15}: {display_v}')
print()
print('JAAS config is prepared (value hidden)')

Kafka
  Bootstrap   : datadosekafka-901-datadosedepiproject001.l.aivencloud.com:15816
  Topic       : DataDose.in
  Username    : ****
  Password    : AVNS****zFPu
  ca.pem path : /Volumes/datadose_workspace/pharma/pharma_certs/ca.pem

Neo4j
  URI      : neo4j+s://95a45d6f.databases.neo4j.io
  User     : ****
  Password : T5GW****EMt0

Snowflake
  host           : TLFCMYB-UJ75221.snowflakecomputing.com
  port           : 443
  sfAccount      : TLFCMYB-UJ75221
  sfUser         : PYSP****_SVC
  sfPassword     : Data****025!
  sfDatabase     : PHARMA_ANALYTICS_DB
  sfSchema       : STAGING
  sfWarehouse    : PHARMA_WH
  sfRole         : PYSPARK_ROLE

JAAS config is prepared (value hidden)


### Step 3 - Neo4j Connection and Helper Functions

In [0]:
# Step 3 - Define Neo4j driver and interaction helpers
import logging
from neo4j import GraphDatabase
from typing import List, Dict

logging.getLogger('neo4j.notifications').setLevel(logging.ERROR)

_neo4j_driver = None

def get_neo4j_driver():
    global _neo4j_driver
    if _neo4j_driver is None:
        _neo4j_driver = GraphDatabase.driver(
            NEO4J_URI,
            auth=(NEO4J_USER, NEO4J_PASSWORD),
            max_connection_pool_size=10,
        )
    return _neo4j_driver


def open_neo4j_driver():
    """Create a brand-new Neo4j driver (no global caching).

    Used inside process_batch/foreachBatch instead of get_neo4j_driver().
    Spark Connect (Shared clusters) pickles the foreachBatch closure to ship
    it to the server - a live driver with an open SSL connection can't be
    pickled (TypeError: cannot pickle 'SSLContext' object). Opening a fresh
    driver per micro-batch and closing it afterward keeps the closure free
    of any live connection object.
    """
    return GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
        max_connection_pool_size=10,
    )


def _empty_interaction() -> Dict[str, str]:
    return {
        'interaction_found'        : 'FALSE',
        'interaction_count'        : '0',
        'interacting_drugs'        : '',
        'interaction_severity'     : '',
        'interaction_type'         : '',
        'shared_ingredient'        : '',
        'ingredient_overlap_count' : '0',
    }


def _worst_severity(severities: List[str]) -> str:
    order = {'Major': 1, 'Moderate': 2, 'Minor': 3}
    ranked = sorted([s for s in severities if s in order], key=lambda x: order[x])
    return ranked[0] if ranked else (severities[0] if severities else '')


def check_interactions(new_drug: str, current_drugs: List[str], driver) -> Dict[str, str]:
    '''Query Neo4j for drug-drug interactions. Returns enrichment fields dict.
    `driver` must be passed in - see open_neo4j_driver() for foreachBatch usage.'''
    if not current_drugs:
        return _empty_interaction()

    interaction_query = '''
        MATCH (d1:Drug)-[r:INTERACTS_WITH]-(d2:Drug)
        WHERE toLower(d1.name) = toLower($new_drug)
          AND ANY(med IN $current_drugs WHERE toLower(d2.name) = toLower(med))
        RETURN d1.name AS drug_a, d2.name AS drug_b,
               r.severity AS severity, r.type AS interaction_type
        ORDER BY CASE r.severity
            WHEN 'Major'    THEN 1
            WHEN 'Moderate' THEN 2
            WHEN 'Minor'    THEN 3
            ELSE 4 END
    '''
    ingredient_query = '''
        MATCH (d1:Drug)-[:HAS_INGREDIENT]->(i:Ingredient)<-[:HAS_INGREDIENT]-(d2:Drug)
        WHERE toLower(d1.name) = toLower($new_drug)
          AND ANY(med IN $current_drugs WHERE toLower(d2.name) = toLower(med))
        RETURN DISTINCT i.name AS ingredient
    '''

    with driver.session() as session:
        interactions = session.run(
            interaction_query, new_drug=new_drug, current_drugs=current_drugs
        ).data()
        shared_ingreds = [
            r['ingredient']
            for r in session.run(
                ingredient_query, new_drug=new_drug, current_drugs=current_drugs
            ).data()
        ]

    if not interactions:
        return {
            **_empty_interaction(),
            'shared_ingredient'        : '|'.join(shared_ingreds),
            'ingredient_overlap_count' : str(len(shared_ingreds)),
        }

    pairs      = [f"{r['drug_a']}\u2194{r['drug_b']}" for r in interactions]
    severities = [r['severity'] for r in interactions if r.get('severity')]
    types      = list({r['interaction_type'] for r in interactions if r.get('interaction_type')})

    return {
        'interaction_found'        : 'TRUE',
        'interaction_count'        : str(len(interactions)),
        'interacting_drugs'        : '|'.join(pairs),
        'interaction_severity'     : _worst_severity(severities),
        'interaction_type'         : '|'.join(types),
        'shared_ingredient'        : '|'.join(shared_ingreds),
        'ingredient_overlap_count' : str(len(shared_ingreds)),
    }


print('Neo4j helper functions defined.')

Neo4j helper functions defined.


#### Debug 3a - Neo4j connection ping

In [0]:
# Debug 3a - Neo4j connection ping
# Expected: Neo4j connected and server datetime
try:
    driver = get_neo4j_driver()
    with driver.session() as session:
        row = session.run("RETURN 'Neo4j connected' AS msg, datetime() AS ts").single()
    print(row['msg'])
    print(f"Server time: {row['ts']}")
except Exception as e:
    print(f'Neo4j connection failed: {e}')
    print('COMMON CAUSES:')
    print('Wrong URI - must start with neo4j+s://')
    print('Wrong user - should be the instance ID (e.g. 403ff197)')
    print('Firewall - AuraDB requires outbound TCP on port 7687')

Neo4j connected
Server time: 2026-07-06T13:23:58.584000000+00:00


#### Debug 3b - Neo4j graph content check

In [0]:
# Debug 3b - Neo4j graph content check
# Expected: Drug node count > 0, INTERACTS_WITH relationships present
try:
    with driver.session() as session:
        node_counts = session.run(
            'MATCH (n) RETURN labels(n)[0] AS label, count(n) AS cnt '
            'ORDER BY cnt DESC LIMIT 10'
        ).data()
        print('Node counts in Neo4j')
        for row in node_counts:
            label_name = row['label'] if row['label'] else 'Unlabeled'
            print(f'   {label_name:<20} : {row["cnt"]:,}')

        relationship_counts = session.run(
            'MATCH ()-[r]->() RETURN type(r) AS rel_type, count(r) AS cnt '
            'ORDER BY cnt DESC LIMIT 5'
        ).data()
        print()
        print('Relationship counts')
        for row in relationship_counts:
            print(f'   {row["rel_type"]:<25} : {row["cnt"]:,}')

        sample_drugs = session.run('MATCH (d:Drug) RETURN d.name AS name LIMIT 5').data()
        print()
        print('Sample Drug nodes')
        for row in sample_drugs:
            print(f'   - {row["name"]}')
except Exception as e:
    print(f'Error: {e}')

Node counts in Neo4j
   Drug                 : 1,952
   Symptom              : 683
   Disease              : 280

Relationship counts
   CAUSES_REACTION           : 6,105
   TREATS                    : 497
   INTERACTS_WITH            : 413

Sample Drug nodes
   - cangrelor
   - capixyl
   - capsaicin + methylsalicylate
   - carbachol
   - carbazochrome


#### Debug 3c - Live interaction lookup test

In [0]:
# Debug 3c - Live interaction lookup dry-run
# Expected: interaction_found=TRUE, severity=Major for warfarin+aspirin
TEST_NEW_DRUG = 'warfarin'
TEST_CURRENT_DRUGS = ['aspirin', 'ibuprofen', 'metformin']

print(f"Testing: '{TEST_NEW_DRUG}' vs {TEST_CURRENT_DRUGS}")
print()

result = check_interactions(TEST_NEW_DRUG, TEST_CURRENT_DRUGS, get_neo4j_driver())
print('check_interactions() result')
for k, v in result.items():
    print(f'   {k:<30} : {v}')

print()
empty = check_interactions('some_drug', [], get_neo4j_driver())
print('Empty current_drugs case')
print(f"   interaction_found : {empty['interaction_found']}   <- should be FALSE")
print(f"   interaction_count : {empty['interaction_count']}       <- should be 0")

Testing: 'warfarin' vs ['aspirin', 'ibuprofen', 'metformin']

check_interactions() result
   interaction_found              : FALSE
   interaction_count              : 0
   interacting_drugs              : 
   interaction_severity           : 
   interaction_type               : 
   shared_ingredient              : 
   ingredient_overlap_count       : 0

Empty current_drugs case
   interaction_found : FALSE   <- should be FALSE
   interaction_count : 0       <- should be 0


### Step 4 - Snowflake Connection and Schema Verification

In [0]:
# Step 4 - Test Snowflake connection
snowflake_identity_df = (
    spark.read
    .format(SNOWFLAKE_SOURCE)
    .options(**sf_options)
    .option('query', '''
        SELECT CURRENT_USER()      AS SF_USER,
               CURRENT_DATABASE()  AS SF_DATABASE,
               CURRENT_WAREHOUSE() AS SF_WAREHOUSE,
               CURRENT_ROLE()      AS SF_ROLE
    ''')
    .load()
)
print('Snowflake connection established.')

Snowflake connection established.


#### Debug 4a - Snowflake identity check

In [0]:
# Debug 4a - Show Snowflake identity
print('Snowflake connection identity')
display(snowflake_identity_df)

expected = {
    'SF_USER'      : 'PYSPARK_SVC',
    'SF_DATABASE'  : 'PHARMA_ANALYTICS_DB',
    'SF_WAREHOUSE' : 'PHARMA_WH',
    'SF_ROLE'      : 'PYSPARK_ROLE',
}
row = snowflake_identity_df.collect()[0]
all_ok = True
for col_name, exp_val in expected.items():
    actual = row[col_name]
    ok = 'OK' if str(actual).upper() == exp_val else 'FAIL'
    if ok == 'FAIL':
        all_ok = False
    print(f'  {ok}  {col_name:<15} : got "{actual}" | expected "{exp_val}"')

if all_ok:
    print()
    print('All Snowflake identity checks passed')

Snowflake connection identity


SF_USER,SF_DATABASE,SF_WAREHOUSE,SF_ROLE
PYSPARK_SVC,PHARMA_ANALYTICS_DB,PHARMA_WH,PYSPARK_ROLE


  OK  SF_USER         : got "PYSPARK_SVC" | expected "PYSPARK_SVC"
  OK  SF_DATABASE     : got "PHARMA_ANALYTICS_DB" | expected "PHARMA_ANALYTICS_DB"
  OK  SF_WAREHOUSE    : got "PHARMA_WH" | expected "PHARMA_WH"
  OK  SF_ROLE         : got "PYSPARK_ROLE" | expected "PYSPARK_ROLE"

All Snowflake identity checks passed


#### Debug 4b - STG_TRANSACTION schema check

In [0]:
# Debug 4b - Confirm STG_TRANSACTION table exists and show schema
# Expected: 26 columns matching the DDL
schema_check_df = (
    spark.read
    .format(SNOWFLAKE_SOURCE)
    .options(**sf_options)
    .option('query', '''
        SELECT COLUMN_NAME, DATA_TYPE
        FROM   INFORMATION_SCHEMA.COLUMNS
        WHERE  TABLE_SCHEMA = 'STAGING'
          AND  TABLE_NAME   = 'STG_TRANSACTION'
        ORDER  BY ORDINAL_POSITION
    ''')
    .load()
)

rows = schema_check_df.collect()
if not rows:
    print('STG_TRANSACTION does not exist in STAGING schema')
    print('Run pharma_snowflake_schema.sql DDL in Snowsight first')
else:
    print(f'STG_TRANSACTION found - {len(rows)} columns:')
    for r in rows:
        print(f'   {r["COLUMN_NAME"]:<35} {r["DATA_TYPE"]}')

STG_TRANSACTION found - 26 columns:
   STG_ID                              NUMBER
   BATCH_ID                            TEXT
   LOAD_TIMESTAMP                      TIMESTAMP_NTZ
   SOURCE_SYSTEM                       TEXT
   IS_PROCESSED                        BOOLEAN
   TX_ID                               TEXT
   PHARMACY                            TEXT
   CITY                                TEXT
   IS_NEW_PRESCRIPTION                 TEXT
   DRUG                                TEXT
   CURRENT_MEDS                        TEXT
   INTERACTION_FOUND                   TEXT
   INTERACTION_COUNT                   TEXT
   INTERACTING_DRUGS                   TEXT
   INTERACTION_SEVERITY                TEXT
   INTERACTION_TYPE                    TEXT
   ACTIVE_INGREDIENT_MATCH             TEXT
   SHARED_INGREDIENT                   TEXT
   INGREDIENT_OVERLAP_COUNT            TEXT
   CURRENT_MEDS_COUNT                  TEXT
   POLYPHARMACY_FLAG                   TEXT
   HIGH_RISK_PATIENT      

### Step 5 - Kafka Structured Streaming Reader

In [0]:
# Step 5 - Define Kafka Structured Streaming reader
# kafka.ssl.truststore.location must point to a UC Volume path (not /tmp or /Workspace) -
# Shared clusters only allow external location or UC Volume paths here.
# kafka.sasl.jaas.config uses the shaded ScramLoginModule class on the Spark classpath.
#
# STARTING_OFFSETS controls whether the stream picks up messages that were sent to the
# topic BEFORE this query starts:
#   'earliest' - read everything currently sitting in DataDose.in (use this while testing
#                with simulator.py, otherwise anything sent before Step 8 starts is invisible
#                and the stream will look like it's "not streaming" even though it's healthy).
#   'latest'   - only read messages produced AFTER the query starts (use this once the
#                pipeline is running continuously in a real deployment).
# IMPORTANT: this only takes effect on a fresh checkpoint. If CHECKPOINT_PATH (Step 8)
# already exists from a previous run, Spark resumes from the committed offset regardless
# of this setting - delete the checkpoint directory (see the cell above Step 8) to force
# a re-read from STARTING_OFFSETS.
STARTING_OFFSETS = get_env('KAFKA_STARTING_OFFSETS', 'earliest')

kafka_stream_df = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers',       KAFKA_BOOTSTRAP)
    .option('kafka.security.protocol',       'SASL_SSL')
    .option('kafka.sasl.mechanism',          'SCRAM-SHA-256')
    .option('kafka.sasl.jaas.config',        KAFKA_JAAS)
    .option('kafka.ssl.truststore.type',     'PEM')
    .option('kafka.ssl.truststore.location', KAFKA_CA_PEM_SPARK)
    .option('subscribe',                     KAFKA_TOPIC)
    .option('kafka.group.id',                KAFKA_GROUP_ID)
    .option('startingOffsets',               STARTING_OFFSETS)
    .option('failOnDataLoss',                'false')
    .option('kafka.request.timeout.ms',      '30000')
    .option('kafka.session.timeout.ms',      '10000')
    .load()
)
print(f'Kafka stream reader defined (not started yet). startingOffsets={STARTING_OFFSETS}, topic={KAFKA_TOPIC!r}')


Kafka stream reader defined (not started yet). startingOffsets=earliest, topic='DataDose.in'


#### Debug 5a - Verify Kafka reader schema

In [0]:
# Debug 5a - Verify Kafka reader schema
print('Kafka stream DataFrame schema')
kafka_stream_df.printSchema()

expected_kafka_cols = {'key','value','topic','partition','offset','timestamp','timestampType'}
missing = expected_kafka_cols - set(kafka_stream_df.columns)
if missing:
    print(f'Missing expected columns: {missing}')
else:
    print('All expected Kafka columns present')

Kafka stream DataFrame schema
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

All expected Kafka columns present


#### Debug 5b - Batch connectivity test (5 real messages)

In [0]:
# Debug 5b - Quick Kafka batch read to confirm real messages arrive
# Ensure simulator.py is running on your local machine first.
# Expected: count > 0, JSON values visible
from pyspark.sql import functions as F

print('Reading up to 5 raw messages from Kafka (batch mode)...')
print('Ensure simulator.py is running: python simulator.py --rate 5')
print(f'Subscribing to topic: {KAFKA_TOPIC!r}')
print()

try:
    kafka_batch_df = (
        spark.read
        .format('kafka')
        .option('kafka.bootstrap.servers',       KAFKA_BOOTSTRAP)
        .option('kafka.security.protocol',       'SASL_SSL')
        .option('kafka.sasl.mechanism',          'SCRAM-SHA-256')
        .option('kafka.sasl.jaas.config',        KAFKA_JAAS)
        .option('kafka.ssl.truststore.type',     'PEM')
        .option('kafka.ssl.truststore.location', KAFKA_CA_PEM_SPARK)
        .option('subscribe',                     KAFKA_TOPIC)
        .option('startingOffsets',               'earliest')
        .option('endingOffsets',                 'latest')
        .option('kafka.request.timeout.ms',      '30000')
        .option('kafka.session.timeout.ms',      '10000')
        .load()
        .limit(5)
    )

    count = kafka_batch_df.count()
    print(f"Messages found in topic '{KAFKA_TOPIC}': {count}")

    if count > 0:
        kafka_batch_df.select(
            'offset', 'partition', 'timestamp',
            F.col('value').cast('string').alias('json_value')
        ).show(5, truncate=100)
    else:
        print('0 messages - is simulator.py running?')
        print('Run: python simulator.py --rate 5')

except Exception as e:
    print(f'Kafka batch read failed: {e}')
    print('COMMON CAUSES:')
    print('ca.pem not copied to the UC Volume (re-run Step 2)')
    print('Wrong SASL password')
    print('spark-sql-kafka JAR not installed on cluster')
    print(f"Topic '{KAFKA_TOPIC}' does not exist in Aiven")

Reading up to 5 raw messages from Kafka (batch mode)...
Ensure simulator.py is running: python simulator.py --rate 5
Subscribing to topic: 'DataDose.in'

Messages found in topic 'DataDose.in': 5
+------+---------+-----------------------+----------------------------------------------------------------------------------------------------+
|offset|partition|              timestamp|                                                                                          json_value|
+------+---------+-----------------------+----------------------------------------------------------------------------------------------------+
|   690|        0|2026-07-05 21:26:21.032|{"transaction_id": 374080, "patient_id": 2835, "pharmacy_id": "PHX_114", "pharmacy_city": "Cairo"...|
|   691|        0|2026-07-05 21:26:21.065|{"transaction_id": 374081, "patient_id": 48314, "pharmacy_id": "PHX_080", "pharmacy_city": "Manso...|
|   692|        0|2026-07-05 21:26:22.066|{"transaction_id": 374082, "patient_id": 47

### Step 6 - JSON Schema and Stream Parsing

In [0]:
# Step 6 - Define JSON schema that matches simulator.py output, parse stream, and split good/bad records
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, ArrayType, LongType
 )

prescription_schema = StructType([
    StructField('transaction_id',  LongType(),              True),
    StructField('patient_id',      IntegerType(),           True),
    StructField('pharmacy_id',     StringType(),            True),
    StructField('pharmacy_city',   StringType(),            True),
    StructField('new_drug',        StringType(),            True),
    StructField('new_drug_dose',   StringType(),            True),
    StructField('new_drug_form',   StringType(),            True),
    StructField('current_drugs',   ArrayType(StringType()), True),
    StructField('patient_age',     IntegerType(),           True),
    StructField('patient_gender',  StringType(),            True),
    StructField('timestamp',       StringType(),            True),
])

raw_parsed_df = (
    kafka_stream_df
    .select(
        F.col('value').cast('string').alias('raw_json'),
        F.col('offset'),
        F.col('timestamp').alias('kafka_timestamp'),
    )
    .withColumn('data', F.from_json(F.col('raw_json'), prescription_schema))
)

# Rows that fail to parse into the schema (malformed JSON, missing required fields)
# come through with data.transaction_id null - split them out instead of silently
# passing garbage into enrichment.
parsed_df = (
    raw_parsed_df
    .filter(F.col('data.transaction_id').isNotNull())
    .select('raw_json', 'offset', 'kafka_timestamp', 'data.*')
)

malformed_df = (
    raw_parsed_df
    .filter(F.col('data.transaction_id').isNull())
    .select('raw_json', 'offset', 'kafka_timestamp')
)

print('Prescription schema and parsed_df/malformed_df defined.')


Prescription schema and parsed_df/malformed_df defined.


#### Debug 6a - Verify parsed schema

In [0]:
# Debug 6a - Verify parsed schema shape
# Expected: 14 cols - 11 simulator fields + raw_json + offset + kafka_timestamp
print('parsed_df schema')
parsed_df.printSchema()

expected_fields = {
    'raw_json', 'offset', 'kafka_timestamp', 'transaction_id', 'patient_id',
    'pharmacy_id', 'pharmacy_city', 'new_drug', 'new_drug_dose',
    'new_drug_form', 'current_drugs', 'patient_age', 'patient_gender', 'timestamp'
}
actual_fields = set(parsed_df.columns)
missing = expected_fields - actual_fields
extra   = actual_fields - expected_fields

if missing: print(f'Missing fields: {missing}')
else:       print('All expected fields present')
if extra:   print(f'Extra fields: {extra}')

# NOTE: malformed_df is built from a streaming source (kafka_stream_df), so actions
# like .count() can't run here directly - Spark only allows that inside writeStream.start()
# (see Step 8b, which streams malformed records to the dead-letter table).
print('malformed_df is a streaming DataFrame - row counts will be visible once Step 8b is running.')


parsed_df schema
root
 |-- raw_json: string (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- transaction_id: long (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- pharmacy_id: string (nullable = true)
 |-- pharmacy_city: string (nullable = true)
 |-- new_drug: string (nullable = true)
 |-- new_drug_dose: string (nullable = true)
 |-- new_drug_form: string (nullable = true)
 |-- current_drugs: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- patient_age: integer (nullable = true)
 |-- patient_gender: string (nullable = true)
 |-- timestamp: string (nullable = true)

All expected fields present
malformed_df is a streaming DataFrame - row counts will be visible once Step 8b is running.


#### Debug 6b - Parse real Kafka messages end-to-end

In [0]:
# Debug 6b - Parse real Kafka messages using prescription_schema
# Expected: all 11 fields populated, no nulls in key columns
print('Parsing real Kafka messages with prescription schema...')

try:
    parsed_batch_df = (
        spark.read
        .format('kafka')
        .option('kafka.bootstrap.servers',       KAFKA_BOOTSTRAP)
        .option('kafka.security.protocol',       'SASL_SSL')
        .option('kafka.sasl.mechanism',          'SCRAM-SHA-256')
        .option('kafka.sasl.jaas.config',        KAFKA_JAAS)
        .option('kafka.ssl.truststore.type',     'PEM')
        .option('kafka.ssl.truststore.location', KAFKA_CA_PEM_SPARK)
        .option('subscribe',                     KAFKA_TOPIC)
        .option('startingOffsets',               'earliest')
        .option('endingOffsets',                 'latest')
        .option('kafka.request.timeout.ms',      '30000')
        .option('kafka.session.timeout.ms',      '10000')
        .load()
        .limit(3)
        .select(F.col('value').cast('string').alias('raw_json'), 'offset')
        .withColumn('data', F.from_json(F.col('raw_json'), prescription_schema))
        .select('raw_json', 'data.*')
    )

    rows = parsed_batch_df.collect()
    print(f'Parsed {len(rows)} message(s)')
    print()

    for i, row in enumerate(rows):
        print(f'Message {i+1}')
        for field in ['transaction_id','patient_id','pharmacy_id','pharmacy_city',
                      'new_drug','new_drug_dose','new_drug_form','current_drugs',
                      'patient_age','patient_gender','timestamp']:
            print(f'  {field:<18} : {row[field]}')
        null_fields = [f for f in ['transaction_id','new_drug','pharmacy_id']
                       if row[f] is None]
        if null_fields:
            print(f'  Null in key fields: {null_fields}')
        else:
            print('  Key fields all populated')
        print()

except Exception as e:
    print(f'Parse test failed: {e}')

Parsing real Kafka messages with prescription schema...
Parsed 3 message(s)

Message 1
  transaction_id     : 374080
  patient_id         : 2835
  pharmacy_id        : PHX_114
  pharmacy_city      : Cairo
  new_drug           : potassium chloride
  new_drug_dose      : 200g
  new_drug_form      : oral_liquid
  current_drugs      : ['epirubicin', 'chlorhexidine vitamin e']
  patient_age        : 23
  patient_gender     : M
  timestamp          : 2026-07-05T21:25:59
  Key fields all populated

Message 2
  transaction_id     : 374081
  patient_id         : 48314
  pharmacy_id        : PHX_080
  pharmacy_city      : Mansoura
  new_drug           : rivaroxaban
  new_drug_dose      : 75%
  new_drug_form      : tablet
  current_drugs      : ['glyceryl monostearate', 'medroxyprogesterone']
  patient_age        : 25
  patient_gender     : M
  timestamp          : 2026-07-05T21:26:21
  Key fields all populated

Message 3
  transaction_id     : 374082
  patient_id         : 47743
  pharmacy_id   

### Step 7 - Define process_batch Enrichment Function

In [0]:
# Step 7 - Define foreachBatch processing function
# Each micro-batch:
 # 1. Collect rows from Kafka
 # 2. Call Neo4j to check drug-drug interactions per row (per-row errors are caught, not fatal)
 # 3. Compute drug risk score and patient risk score
 # 4. Write enriched rows to Snowflake STAGING.STG_TRANSACTION
 # 5. Anything that fails at any step (bad row, Snowflake write) is written directly
 #    into Snowflake STAGING.STG_TRANSACTION as a dead-letter record instead of being dropped.
from datetime import datetime, UTC
from pyspark.sql import DataFrame
import json as _json


def write_dead_letter(records: list, failure_reason: str, batch_id: int) -> None:
    """Append failed records directly into Snowflake STAGING.STG_TRANSACTION. Never raises -
    a dead-letter write failure is logged, not fatal, so it can't take the
    stream down on top of whatever already went wrong."""
    if not records:
        return
    try:
        dead_letter_rows = [
            {
                'BATCH_ID'        : str(batch_id),
                'SOURCE_SYSTEM'   : f'DEAD_LETTER_{failure_reason.upper()}',
                'TX_ID'           : str(r.get('transaction_id') or r.get('TX_ID') or ''),
                'DRUG'            : '',
                'RAW_RECORD'      : _json.dumps(r, ensure_ascii=False, default=str),
                'LOAD_TIMESTAMP'  : datetime.now(UTC),
                'IS_PROCESSED'    : 'FALSE',
            }
            for r in records
        ]
        (
            spark.createDataFrame(dead_letter_rows).write
            .format(SNOWFLAKE_SOURCE)
            .options(**sf_options)
            .option('dbtable', 'STAGING.STG_TRANSACTION')
            .option('column_mapping', 'name')
            .option('column_mismatch_behavior', 'ignore')
            .mode('append')
            .save()
        )
        print(f'   Batch {batch_id}: {len(records)} record(s) written to Snowflake dead letter ({failure_reason})')
    except Exception as e:
        print(f'   Batch {batch_id}: DEAD LETTER WRITE ALSO FAILED ({failure_reason}): {e}')
        print(f'   Lost records: {records}')


def process_batch(batch_df: DataFrame, batch_id: int) -> None:
    rows = batch_df.collect()
    if not rows:
        print(f'   Batch {batch_id}: empty, skipping.')
        return

    offsets = [r['offset'] for r in rows if r['offset'] is not None]
    # Offset-range batch id instead of a random uuid, so a checkpoint replay of the
    # same offset range is identifiable/deduplicable downstream instead of silently
    # producing a fresh-looking duplicate batch.
    batch_uuid = f'KAFKA-{batch_id}-{min(offsets)}-{max(offsets)}' if offsets else f'KAFKA-{batch_id}-NOOFFSET'
    enriched_rows = []
    failed_rows = []
    skipped = 0

    # Fresh driver per micro-batch (not the cached global from get_neo4j_driver()) -
    # keeps this function's closure free of live SSL connection objects, which
    # Spark Connect can't pickle when shipping foreachBatch to the server.
    neo4j_driver = open_neo4j_driver()

    # Computed once per batch and sent explicitly as real columns below, instead of
    # being left for the Snowflake connector to fill in. See the FIX note above the
    # Snowflake write for why this matters.
    #
    # FIX: this must stay a real timezone-aware datetime object, NOT a formatted
    # string. A string forces Snowflake's COPY INTO to text-parse it against the
    # target column's expected timestamp format (TIMESTAMP_TZ/_LTZ expect a
    # timezone offset baked into the string) - mismatches there raise
    # "Can't parse '...' as timestamp with format '...'". A native datetime lets
    # Spark infer a proper TimestampType column, which the connector serializes
    # correctly regardless of whether the Snowflake column is _TZ, _LTZ, or _NTZ.
    load_timestamp = datetime.now(UTC)

    for row in rows:
        try:
            new_drug = row['new_drug'] or ''
            current_drugs = [d for d in (row['current_drugs'] or []) if d]

            neo4j_result = check_interactions(new_drug, current_drugs, neo4j_driver)
            current_meds_count = len(current_drugs)
            polypharmacy_flag = 'TRUE' if current_meds_count >= 5 else 'FALSE'
            interaction_count = int(neo4j_result['interaction_count'])
            severity = neo4j_result['interaction_severity']
            overlap_count = int(neo4j_result['ingredient_overlap_count'])

            severity_score = {'Major': 40, 'Moderate': 20, 'Minor': 10}.get(severity, 0)
            drug_risk_score = min(100.0, round(
                severity_score
                + interaction_count * 5
                + overlap_count * 3
                + (5 if polypharmacy_flag == 'TRUE' else 0),
                2
            ))
            age_val = row['patient_age'] if row['patient_age'] is not None else 40
            age_multiplier = 1.0 + (age_val - 40) * 0.005
            patient_risk_score = min(100.0, round(drug_risk_score * age_multiplier, 2))
            high_risk_patient = 'TRUE' if patient_risk_score >= 60 else 'FALSE'
            interaction_rate = round(interaction_count / max(1, current_meds_count), 4)

            enriched_rows.append({
                'BATCH_ID'                 : batch_uuid,
                'SOURCE_SYSTEM'            : 'AIVEN_KAFKA',
                'TX_ID'                    : str(row['transaction_id']),
                'PHARMACY'                 : row['pharmacy_id']   or '',
                'CITY'                     : row['pharmacy_city'] or '',
                'IS_NEW_PRESCRIPTION'      : 'New',
                'DRUG'                     : new_drug,
                'CURRENT_MEDS'             : '|'.join(current_drugs),
                'INTERACTION_FOUND'        : neo4j_result['interaction_found'],
                'INTERACTION_COUNT'        : neo4j_result['interaction_count'],
                'INTERACTING_DRUGS'        : neo4j_result['interacting_drugs'],
                'INTERACTION_SEVERITY'     : neo4j_result['interaction_severity'],
                'INTERACTION_TYPE'         : neo4j_result['interaction_type'],
                'ACTIVE_INGREDIENT_MATCH'  : 'TRUE' if overlap_count > 0 else 'FALSE',
                'SHARED_INGREDIENT'        : neo4j_result['shared_ingredient'],
                'INGREDIENT_OVERLAP_COUNT' : neo4j_result['ingredient_overlap_count'],
                'CURRENT_MEDS_COUNT'       : str(current_meds_count),
                'POLYPHARMACY_FLAG'        : polypharmacy_flag,
                'HIGH_RISK_PATIENT'        : high_risk_patient,
                'DRUG_RISK_SCORE'          : str(drug_risk_score),
                'PATIENT_RISK_SCORE'       : str(patient_risk_score),
                'INTERACTION_RATE'         : str(interaction_rate),
                'RAW_RECORD'               : row['raw_json'],
                # FIX: sent explicitly instead of being left for the Snowflake connector
                # to null-fill (see note on the write below). IS_PROCESSED starts FALSE
                # and is expected to be flipped by a downstream consumer/job.
                'LOAD_TIMESTAMP'           : load_timestamp,
                'IS_PROCESSED'             : 'FALSE',
            })
        except Exception as e:
            skipped += 1
            print(f'   Row failed in batch {batch_id}, tx_id={row["transaction_id"]}: {e}')
            failed_rows.append({'transaction_id': row['transaction_id'], 'raw_json': row['raw_json'], 'error': str(e)})
            continue

    neo4j_driver.close()

    if failed_rows:
        write_dead_letter(failed_rows, 'enrichment_failed', batch_id)

    if not enriched_rows:
        print(f'   Batch {batch_id}: all {skipped} row(s) failed enrichment, nothing written.')
        return

    enriched_df = spark.createDataFrame(enriched_rows)

    # Write is wrapped so a transient Snowflake blip logs and moves on
    # instead of crashing the whole streaming query. Since checkpointLocation
    # advances past this batch either way, a write failure here means that
    # batch's data is lost rather than retried - acceptable tradeoff for keeping
    # the stream alive, but worth alerting on if these prints ever show up.
    snowflake_ok = False

    try:
        # FIX: STG_TRANSACTION has 26 columns. STG_ID is an IDENTITY column,
        # LOAD_TIMESTAMP and IS_PROCESSED have table-side DEFAULTs - the previous
        # version of this code left all three out of enriched_df and relied on
        # 'column_mismatch_behavior'='ignore' to paper over the gap. That option
        # does NOT fall back to the column's DEFAULT/IDENTITY the way the old
        # comment assumed - per Snowflake's own docs, the connector explicitly
        # inserts NULL into any column missing from the DataFrame. STG_ID is
        # NOT NULL (it's the identity primary key), so every single batch write
        # was failing with a NULL-into-NOT-NULL-identity-column error - caught
        # below and silently rerouted to the dead-letter path, which is why
        # nothing ever landed in Snowflake even though everything upstream
        # looked healthy.
        #
        # Fix: LOAD_TIMESTAMP and IS_PROCESSED are now computed above and sent
        # as real columns (see enriched_rows), so STG_ID is the ONLY remaining
        # column left for the connector to fill in - and since it's an IDENTITY
        # column, Snowflake auto-generates it correctly when no value is
        # supplied for it at all (as opposed to being sent an explicit NULL).
        #
        # 'columnmap' would normally be the cleaner way to state this explicitly,
        # but Databricks SERVERLESS compute blocks that option entirely
        # (DATA_SOURCE_OPTIONS_VALIDATION_FAILED / SERVERLESS_WRITE_OPTIONS_NOT_ALLOWED).
        # 'column_mapping'='name' + 'column_mismatch_behavior'='ignore' is the
        # Serverless-safe equivalent, and is now only covering STG_ID.
        (
            enriched_df.write
            .format(SNOWFLAKE_SOURCE)
            .options(**sf_options)
            .option('dbtable', 'STAGING.STG_TRANSACTION')
            .option('column_mapping', 'name')
            .option('column_mismatch_behavior', 'ignore')
            .mode('append')
            .save()
        )
        snowflake_ok = True
    except Exception as e:
        # FIX: print the full exception (type + message + any chained cause) instead
        # of the bare f-string, since Snowflake/JDBC errors often bury the actually
        # useful detail (e.g. the specific column and constraint) a level or two down.
        print(f'   Batch {batch_id}: Snowflake write FAILED: {type(e).__name__}: {e}')
        cause = getattr(e, '__cause__', None)
        if cause:
            print(f'      Caused by: {type(cause).__name__}: {cause}')
        write_dead_letter(enriched_rows, 'snowflake_write_failed', batch_id)

    interaction_count_total = sum(1 for r in enriched_rows if r['INTERACTION_FOUND'] == 'TRUE')
    high_risk_count = sum(1 for r in enriched_rows if r['HIGH_RISK_PATIENT'] == 'TRUE')
    print(f'  Batch {batch_id} [{batch_uuid}] -> {len(enriched_rows)} enriched, {skipped} skipped '
          f'| Snowflake={"OK" if snowflake_ok else "FAILED"}')
    print(f'     Interactions detected : {interaction_count_total}')
    print(f'     High-risk patients    : {high_risk_count}')


print('process_batch function defined (Snowflake-only sink, per-row error handling, dead-letter into Snowflake on failure).')


process_batch function defined (Snowflake-only sink, per-row error handling, dead-letter into Snowflake on failure).


#### Debug 7 - Dry-run process_batch on 5 real Kafka messages

In [0]:
# Debug 7 - Dry-run process_batch on 5 real Kafka messages
# Reads from Kafka as batch, runs full enrichment, writes to Snowflake
# Rows will have BATCH_ID = 'KAFKA-9999-*' so they are easy to identify
# Expected: rows in Snowflake with interaction/risk fields populated
print('Running dry-run of process_batch on 5 real Kafka messages...')
print('This will write to Snowflake STAGING.STG_TRANSACTION (BATCH_ID=KAFKA-9999-*)')
print()

try:
    dry_run_df = (
        spark.read
        .format('kafka')
        .option('kafka.bootstrap.servers',       KAFKA_BOOTSTRAP)
        .option('kafka.security.protocol',       'SASL_SSL')
        .option('kafka.sasl.mechanism',          'SCRAM-SHA-256')
        .option('kafka.sasl.jaas.config',        KAFKA_JAAS)
        .option('kafka.ssl.truststore.type',     'PEM')
        .option('kafka.ssl.truststore.location', KAFKA_CA_PEM_SPARK)
        .option('subscribe',                     KAFKA_TOPIC)
        .option('startingOffsets',               'earliest')
        .option('endingOffsets',                 'latest')
        .option('kafka.request.timeout.ms',      '30000')
        .option('kafka.session.timeout.ms',      '10000')
        .load()
        .limit(5)
        .select(F.col('value').cast('string').alias('raw_json'), 'offset', 'timestamp')
        .withColumn('data', F.from_json(F.col('raw_json'), prescription_schema))
        .select('raw_json', 'offset', F.col('timestamp').alias('kafka_timestamp'), 'data.*')
    )

    process_batch(dry_run_df, batch_id=9999)

    print()
    print('Verifying rows written to Snowflake')
    df_written = (
        spark.read
        .format(SNOWFLAKE_SOURCE)
        .options(**sf_options)
        .option('query', '''
            SELECT TX_ID, DRUG, CITY, INTERACTION_FOUND,
                   INTERACTION_SEVERITY, HIGH_RISK_PATIENT,
                   PATIENT_RISK_SCORE, BATCH_ID
            FROM   STAGING.STG_TRANSACTION
            WHERE  BATCH_ID LIKE 'KAFKA-9999-%'
            ORDER  BY LOAD_TIMESTAMP DESC
        ''')
        .load()
    )
    df_written.show(truncate=False)

    count = df_written.count()
    if count > 0:
        print(f'{count} row(s) confirmed in Snowflake')
    else:
        print('0 rows - check Snowflake permissions or Kafka message availability')

except Exception as e:
    print(f'Dry-run failed: {e}')

Running dry-run of process_batch on 5 real Kafka messages...
This will write to Snowflake STAGING.STG_TRANSACTION (BATCH_ID=KAFKA-9999-*)

  Batch 9999 [KAFKA-9999-690-694] -> 5 enriched, 0 skipped | Snowflake=OK
     Interactions detected : 0
     High-risk patients    : 0

Verifying rows written to Snowflake
+------+-------------------+--------+-----------------+--------------------+-----------------+------------------+------------------+
|TX_ID |DRUG               |CITY    |INTERACTION_FOUND|INTERACTION_SEVERITY|HIGH_RISK_PATIENT|PATIENT_RISK_SCORE|BATCH_ID          |
+------+-------------------+--------+-----------------+--------------------+-----------------+------------------+------------------+
|374083|alginic acid       |Mansoura|FALSE            |                    |FALSE            |0.0               |KAFKA-9999-690-694|
|374081|rivaroxaban        |Mansoura|FALSE            |                    |FALSE            |0.0               |KAFKA-9999-690-694|
|374084|ethosuximide   

### Step 8 - Start Live Streaming Pipeline
Only run this after all debug cells above have passed.

**Serverless note:** Databricks serverless compute does not support the continuous
`processingTime` trigger (`[INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED]`). Serverless only
allows `trigger(availableNow=True)` (or `trigger(once=True)`), which processes everything
currently sitting in Kafka in one or more micro-batches and then **stops on its own** -
it does not keep running/polling in the background like `processingTime` does.

To get near-continuous behavior on serverless:
- Re-run the Step 8 / Step 8b cells periodically (manually, or via a Databricks Job
  scheduled every N minutes) to pick up newly-arrived Kafka messages each run, **or**
- Move this notebook to a classic (non-serverless) all-purpose/job cluster, where
  `trigger(processingTime='10 seconds')` works unchanged and the stream truly runs forever.

The cells below use `availableNow=True` so they work on serverless as-is.


#### Reset checkpoints (run only if you need a clean re-read from STARTING_OFFSETS)

In [0]:
# Optional - Reset checkpoints before Step 8
# Structured Streaming resumes from the committed offset in CHECKPOINT_PATH / MALFORMED_CHECKPOINT_PATH
# on every run, regardless of the 'startingOffsets' option. If you've run this pipeline before and it
# now looks like "nothing streams", it may simply be replaying/waiting from an old committed offset
# with no new messages after it. Uncomment and run this ONCE to force a clean re-read, then re-run
# Step 8 and Step 8b below.

# dbutils.fs.rm('/Volumes/ali_vm/pharma/pharma_pipeline_data/checkpoints/kafka_to_snowflake_and_out', recurse=True)
# dbutils.fs.rm('/Volumes/ali_vm/pharma/pharma_pipeline_data/checkpoints/malformed_dead_letter', recurse=True)
# print('Checkpoints cleared - Step 8 / Step 8b will re-read from STARTING_OFFSETS on next start.')
print('Checkpoint reset cell ready - uncomment the two dbutils.fs.rm lines above if needed.')


Checkpoint reset cell ready - uncomment the two dbutils.fs.rm lines above if needed.


In [0]:
# Step 8 - Start the live Kafka -> Neo4j -> Snowflake + DataDose.out streaming pipeline
# NOTE: Serverless compute does not support trigger(processingTime=...) - it only supports
# trigger(availableNow=True) / trigger(once=True). availableNow=True drains everything
# currently available on the Kafka topic (in micro-batches sized by maxOffsetsPerTrigger,
# if set) and then the query STOPS itself - query.isActive will become False once done.
# Re-run this cell (or schedule it as a Job) to pick up newly-arrived messages.
CHECKPOINT_PATH = "/Volumes/datadose_workspace/pharma/pharma_pipeline_data/checkpoints/kafka_to_snowflake_and_out"
query = (
    parsed_df.writeStream
    .foreachBatch(process_batch)
    .option('checkpointLocation', CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start()
)

print('Streaming pipeline STARTED (availableNow - will process then stop)')
print(f'   Query ID     : {query.id}')
print(f'   Checkpoint   : {CHECKPOINT_PATH}')
print(f'   Trigger      : availableNow (serverless-compatible; drains available data, then stops)')
print(f'   Input topic  : {KAFKA_TOPIC}')
print(f'   Output topic : {KAFKA_TOPIC_OUT}')
print()
print('Use Debug 8a/8b/8c/8d cells below to monitor the stream.')
print('Re-run this cell whenever you want to pick up new Kafka messages, or schedule')
print('it as a Databricks Job on a recurring interval for near-continuous processing.')


Streaming pipeline STARTED (availableNow - will process then stop)
   Query ID     : bcd27039-2d21-405d-96cb-0ea937807cd1
   Checkpoint   : /Volumes/datadose_workspace/pharma/pharma_pipeline_data/checkpoints/kafka_to_snowflake_and_out
   Trigger      : availableNow (serverless-compatible; drains available data, then stops)
   Input topic  : DataDose.in
   Output topic : DataDose.out

Use Debug 8a/8b/8c/8d cells below to monitor the stream.
Re-run this cell whenever you want to pick up new Kafka messages, or schedule
it as a Databricks Job on a recurring interval for near-continuous processing.


#### Step 8b - Start dead-letter stream for malformed Kafka messages
Runs alongside the main pipeline. Messages that fail schema parsing (see Step 6) never reach `process_batch` - this catches them separately so nothing from `DataDose.in` is silently dropped.

Like Step 8, this uses `trigger(availableNow=True)` for serverless compatibility - it drains
currently-available malformed messages and then stops; re-run alongside Step 8 to keep up.


In [0]:
# Step 8b - Stream malformed (unparseable) records straight into Snowflake STAGING.STG_TRANSACTION
# NOTE: Same serverless constraint as Step 8 - trigger(processingTime=...) is not supported,
# so this uses trigger(availableNow=True) as well. It will drain available malformed
# records and then stop on its own (malformed_query.isActive becomes False when done).
def write_malformed_batch(batch_df: DataFrame, batch_id: int) -> None:
    rows = batch_df.collect()
    if not rows:
        return
    records = [
        {'offset': r['offset'], 'kafka_timestamp': str(r['kafka_timestamp']), 'raw_json': r['raw_json']}
        for r in rows
    ]
    write_dead_letter(records, 'schema_parse_failed', batch_id)


MALFORMED_CHECKPOINT_PATH = "/Volumes/datadose_workspace/pharma/pharma_pipeline_data/checkpoints/malformed_dead_letter"
malformed_query = (
    malformed_df.writeStream
    .foreachBatch(write_malformed_batch)
    .option('checkpointLocation', MALFORMED_CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start()
)

print('Dead-letter stream for malformed records STARTED (availableNow - will process then stop)')
print(f'   Query ID   : {malformed_query.id}')
print(f'   Checkpoint : {MALFORMED_CHECKPOINT_PATH}')
print('   Sink       : Snowflake STAGING.STG_TRANSACTION (reason=schema_parse_failed)')
print('Re-run alongside Step 8 whenever you want to pick up new malformed messages.')


Dead-letter stream for malformed records STARTED (availableNow - will process then stop)
   Query ID   : 0d836c5c-4459-4cda-bac1-4f1787d8a766
   Checkpoint : /Volumes/datadose_workspace/pharma/pharma_pipeline_data/checkpoints/malformed_dead_letter
   Sink       : Snowflake STAGING.STG_TRANSACTION (reason=schema_parse_failed)
Re-run alongside Step 8 whenever you want to pick up new malformed messages.


#### Debug 8e - Verify DataDose.out is receiving enriched messages

In [0]:
# Debug 8e - Batch-read DataDose.out to confirm the live pipeline is actually publishing
# downstream. Run this AFTER Step 8 has been running for at least one trigger interval (10s+).
# This is the mirror of Debug 5b, but for the OUTPUT topic - Debug 5b only ever proved
# messages exist on KAFKA_TOPIC (in); it says nothing about whether process_batch's
# Kafka-out write is succeeding.
print(f'Reading up to 5 messages from {KAFKA_TOPIC_OUT!r} (batch mode)...')

try:
    kafka_out_check_df = (
        spark.read
        .format('kafka')
        .option('kafka.bootstrap.servers',       KAFKA_BOOTSTRAP)
        .option('kafka.security.protocol',       'SASL_SSL')
        .option('kafka.sasl.mechanism',          'SCRAM-SHA-256')
        .option('kafka.sasl.jaas.config',        KAFKA_JAAS)
        .option('kafka.ssl.truststore.type',     'PEM')
        .option('kafka.ssl.truststore.location', KAFKA_CA_PEM_SPARK)
        .option('subscribe',                     KAFKA_TOPIC_OUT)
        .option('startingOffsets',               'earliest')
        .option('endingOffsets',                 'latest')
        .option('kafka.request.timeout.ms',      '30000')
        .option('kafka.session.timeout.ms',      '10000')
        .load()
        .limit(5)
    )

    count = kafka_out_check_df.count()
    print(f"Messages found in topic {KAFKA_TOPIC_OUT!r}: {count}")

    if count > 0:
        kafka_out_check_df.select(
            'offset', 'partition', 'timestamp',
            F.col('value').cast('string').alias('json_value')
        ).show(5, truncate=100)
    else:
        print('0 messages on the OUT topic. If the IN topic has messages but OUT is empty:')
        print('  - Check the printed "kafka_out=FAILED" lines from process_batch (Step 8/Debug 8b)')
        print('  - Confirm KAFKA_TOPIC_OUT matches the topic name exactly (case-sensitive,')
        print('    no stray whitespace - see the quoted print in Step 1)')
        print('  - Confirm the Aiven ACL for this user/service allows WRITE on DataDose.out,')
        print('    not just READ/WRITE on DataDose.in')

except Exception as e:
    print(f'Kafka batch read of {KAFKA_TOPIC_OUT!r} failed: {e}')


Reading up to 5 messages from 'DataDose.out' (batch mode)...
Messages found in topic 'DataDose.out': 5


26/07/06 13:38:53 Spark Server has not sent updates for Streaming Query fec28ecf-3cb4-4f18-bac4-fac3ffd2561f in 60 seconds, but the query is still active. Marking query as in-progress. Spark Session ID is 85b70e33-6ed9-44c7-a788-070f23e14a9f. This is typically not a problem.


+------+---------+-----------------------+----------------------------------------------------------------------------------------------------+
|offset|partition|              timestamp|                                                                                          json_value|
+------+---------+-----------------------+----------------------------------------------------------------------------------------------------+
|     0|        0|2026-07-05 23:34:40.854|{"BATCH_ID": "KAFKA-9999-690-694", "SOURCE_SYSTEM": "AIVEN_KAFKA", "TX_ID": "374083", "PHARMACY":...|
|     1|        0|2026-07-05 23:34:40.855|{"BATCH_ID": "KAFKA-9999-690-694", "SOURCE_SYSTEM": "AIVEN_KAFKA", "TX_ID": "374082", "PHARMACY":...|
|     2|        0|2026-07-05 23:34:40.855|{"BATCH_ID": "KAFKA-9999-690-694", "SOURCE_SYSTEM": "AIVEN_KAFKA", "TX_ID": "374080", "PHARMACY":...|
|     3|        0|2026-07-05 23:34:40.855|{"BATCH_ID": "KAFKA-9999-690-694", "SOURCE_SYSTEM": "AIVEN_KAFKA", "TX_ID": "374081", "PHARMAC

### Step 9 - Monitor and Stop Stream

#### Debug 8a - Stream status
With `availableNow=True`, the query is expected to finish and become inactive once it has
drained everything available - `isActive=False` here means "done processing", not "broken".


In [0]:
# Debug 8a - Check stream status (run any time after Step 8)
# Expected: with availableNow, isActive may already be False if the drain finished quickly -
# that's success, not failure. Check lastProgress for what it actually processed.
import time

print(f'Stream active : {query.isActive}')
print(f'Status        : {query.status}')
print()

if query.lastProgress:
    p = query.lastProgress
    print('Last progress')
    print(f"  Batch ID           : {p.get('batchId', '-')}")
    print(f"  Input rows/sec     : {p.get('inputRowsPerSecond', '-')}")
    print(f"  Processed rows/sec : {p.get('processedRowsPerSecond', '-')}")
    sources = p.get('sources', [{}])
    if sources:
        print(f"  Kafka end offset   : {sources[0].get('endOffset', '-')}")
else:
    print('No progress yet - waiting for first batch...')

if not query.isActive:
    print()
    print('Query has stopped (expected with availableNow once it drains available data).')
    print('Re-run Step 8 to process any newly-arrived Kafka messages.')


Stream active : True
Status        : {'message': 'Writing offsets to log', 'isDataAvailable': False, 'isTriggerActive': True}

No progress yet - waiting for first batch...


#### Debug 8b - Poll for new batches over 60 seconds
With `availableNow=True` the original query typically finishes fast. This cell re-runs
Step 8 on a short interval instead of watching one long-lived query, to approximate the
old "watch it run for 60 seconds" check on serverless.


In [0]:
# Debug 8b - Poll for new batches for 60 seconds (6 checks x 10s)
# Expected: batch counts increasing across re-runs, rows being written to Snowflake.
# Since availableNow queries stop once drained, "watching one query" isn't meaningful -
# instead this re-starts a fresh availableNow query each interval to pick up new data.
print('Polling for new data every 10 seconds (6 checks)...')
print('Ensure simulator.py is running!\n')

for i in range(6):
    time.sleep(10)

    if not query.isActive:
        query = (
            parsed_df.writeStream
            .foreachBatch(process_batch)
            .option('checkpointLocation', CHECKPOINT_PATH)
            .trigger(availableNow=True)
            .start()
        )
        query.awaitTermination()

    prog  = query.lastProgress or {}
    batch = prog.get('batchId', '-')
    rows_s = prog.get('inputRowsPerSecond', '-')
    print(
        f'  [{i+1}/6]  active={query.isActive}  '
        f'batch={batch}  input_rows/s={rows_s}'
    )


Polling for new data every 10 seconds (6 checks)...
Ensure simulator.py is running!



26/07/06 13:39:18 Spark Server has not sent updates for Streaming Query fec28ecf-3cb4-4f18-bac4-fac3ffd2561f in 60 seconds, but the query is still active. Marking query as in-progress. Spark Session ID is 85b70e33-6ed9-44c7-a788-070f23e14a9f. This is typically not a problem.


  [1/6]  active=True  batch=-  input_rows/s=-
  [2/6]  active=True  batch=-  input_rows/s=-


26/07/06 13:39:44 Spark Server has not sent updates for Streaming Query fec28ecf-3cb4-4f18-bac4-fac3ffd2561f in 60 seconds, but the query is still active. Marking query as in-progress. Spark Session ID is 85b70e33-6ed9-44c7-a788-070f23e14a9f. This is typically not a problem.


  [3/6]  active=True  batch=-  input_rows/s=-
  [4/6]  active=True  batch=-  input_rows/s=-
  [5/6]  active=True  batch=-  input_rows/s=-


26/07/06 13:40:09 Spark Server has not sent updates for Streaming Query fec28ecf-3cb4-4f18-bac4-fac3ffd2561f in 60 seconds, but the query is still active. Marking query as in-progress. Spark Session ID is 85b70e33-6ed9-44c7-a788-070f23e14a9f. This is typically not a problem.


  [6/6]  active=True  batch=-  input_rows/s=-


#### Debug 8c - Latest rows written to Snowflake

In [0]:
# Debug 8c - Read latest rows written to Snowflake by the live stream
# Expected: recent rows with BATCH_ID not starting with 'KAFKA-9999-'
live_stream_df = (
    spark.read
    .format(SNOWFLAKE_SOURCE)
    .options(**sf_options)
    .option('query', '''
        SELECT TX_ID, DRUG, CITY,
               INTERACTION_FOUND, INTERACTION_SEVERITY,
               HIGH_RISK_PATIENT, PATIENT_RISK_SCORE,
               POLYPHARMACY_FLAG, BATCH_ID, LOAD_TIMESTAMP
        FROM   STAGING.STG_TRANSACTION
        WHERE  BATCH_ID NOT LIKE 'KAFKA-9999-%'
        ORDER  BY LOAD_TIMESTAMP DESC
        LIMIT  20
    ''')
    .load()
)

count = live_stream_df.count()
print(f'{count} live stream row(s) in Snowflake')
live_stream_df.show(truncate=False)

20 live stream row(s) in Snowflake
+------+----------------------+-------------------+-----------------+--------------------+-----------------+------------------+-----------------+--------------------+--------------------------+
|TX_ID |DRUG                  |CITY               |INTERACTION_FOUND|INTERACTION_SEVERITY|HIGH_RISK_PATIENT|PATIENT_RISK_SCORE|POLYPHARMACY_FLAG|BATCH_ID            |LOAD_TIMESTAMP            |
+------+----------------------+-------------------+-----------------+--------------------+-----------------+------------------+-----------------+--------------------+--------------------------+
|387826|crotamiton            |Zagazig            |FALSE            |                    |FALSE            |0.0               |FALSE            |KAFKA-13-14429-14482|2026-07-06 01:16:30.753802|
|387828|apiole                |6th of October City|FALSE            |                    |FALSE            |0.0               |FALSE            |KAFKA-13-14429-14482|2026-07-06 01:16:30.753

#### Debug 8d - Pipeline health summary

In [0]:
# Debug 8d - Summary stats from Snowflake (final health check)
# Expected: total records, interaction rate, high-risk count, avg risk score
pipeline_stats_df = (
    spark.read
    .format(SNOWFLAKE_SOURCE)
    .options(**sf_options)
    .option('query', '''
        SELECT
            COUNT(*)                                                     AS TOTAL_RECORDS,
            SUM(CASE WHEN INTERACTION_FOUND = 'TRUE' THEN 1 ELSE 0 END) AS INTERACTIONS_DETECTED,
            SUM(CASE WHEN HIGH_RISK_PATIENT = 'TRUE' THEN 1 ELSE 0 END) AS HIGH_RISK_PATIENTS,
            SUM(CASE WHEN POLYPHARMACY_FLAG = 'TRUE' THEN 1 ELSE 0 END) AS POLYPHARMACY_CASES,
            ROUND(AVG(PATIENT_RISK_SCORE::FLOAT), 2)                    AS AVG_RISK_SCORE,
            MAX(LOAD_TIMESTAMP)                                          AS LAST_WRITE
        FROM STAGING.STG_TRANSACTION
    ''')
    .load()
)

print('Pipeline Health Summary')
pipeline_stats_df.show(truncate=False)

Pipeline Health Summary
+-------------+---------------------+------------------+------------------+--------------+--------------------------+
|TOTAL_RECORDS|INTERACTIONS_DETECTED|HIGH_RISK_PATIENTS|POLYPHARMACY_CASES|AVG_RISK_SCORE|LAST_WRITE                |
+-------------+---------------------+------------------+------------------+--------------+--------------------------+
|12238        |5                    |0                 |2063              |0.91          |2026-07-06 13:27:26.085768|
+-------------+---------------------+------------------+------------------+--------------+--------------------------+



#### Stop the stream

In [0]:
# Step 9 - Stop both streaming queries (safe to run even if they already finished on their own)
if 'query' in globals() and query.isActive:
    query.stop()
    print('Main pipeline stream stopped.')
else:
    print('Main pipeline stream already inactive (expected with availableNow once drained).')
print(f'   Final status : {query.status}')

if 'malformed_query' in globals():
    if malformed_query.isActive:
        malformed_query.stop()
        print('Dead-letter (malformed) stream stopped.')
    else:
        print('Dead-letter (malformed) stream already inactive (expected with availableNow once drained).')
    print(f'   Final status : {malformed_query.status}')


26/07/06 13:40:34 Spark Server has not sent updates for Streaming Query fec28ecf-3cb4-4f18-bac4-fac3ffd2561f in60 seconds, and the query is no longer active. Marking query as finished. Spark Session ID is 85b70e33-6ed9-44c7-a788-070f23e14a9f. This is typically not a problem.


Main pipeline stream stopped.
   Final status : {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
Dead-letter (malformed) stream already inactive (expected with availableNow once drained).
   Final status : {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
